# Jour 4 : Deep Learning, Évaluation rigoureuse et Production

## Phase 0 : Mise en route

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import flask
import streamlit

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"joblib     : {joblib.__version__}")
print(f"Flask      : {flask.__version__}")
print(f"Streamlit  : {streamlit.__version__}")
print("Tout est prêt !")

TensorFlow : 2.21.0
Keras      : 3.14.1
joblib     : 1.5.3
Flask      : 3.1.3
Streamlit  : 1.58.0
Tout est prêt !


C:\Users\amosc\AppData\Local\Temp\ipykernel_18888\3742180843.py:11: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"Flask      : {flask.__version__}")


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.svm import SVC

## Phase 1 : Séparer les données proprement, train / validation / test

In [7]:
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """Découpe X, y en trois jeux : train, validation, test.
    Doit renvoyer 6 objets : X_train, X_val, X_test, y_train, y_val, y_test.
    Les proportions doivent rester respectées (utiliser stratify=y).
    """
    if val_size <= 0:
        raise ValueError("val_size doit être > 0")

    # premier split pour isoler le test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # recalcul de la proportion val sur le reste
    # ex : val=0.2 sur 569 lignes = 114 lignes
    # le reste après test = 455 lignes → val_ajuste = 114/455 = 0.25
    val_size_ajuste = val_size / (1 - test_size)

    # deuxième split sur le reste pour isoler la validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size_ajuste,
        random_state=random_state,
        stratify=y_temp
    )

    # renvoyer les 6 objets
    print(f"Train      : {len(X_train)} lignes")
    print(f"Validation : {len(X_val)} lignes")
    print(f"Test       : {len(X_test)} lignes")
    print(f"Total      : {len(X_train)+len(X_val)+len(X_test)} (original : {len(X)})")
    print()

    # stratify=y : garder la même proportion de classes dans chaque jeu
    # qu'au départ, pour ne pas créer un test sans la classe rare
    for nom, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
        pct = yy.mean() * 100
        print(f"  {nom:<6} : classe 1 = {pct:.1f}%")

    print("\nRépartition des classes conservée dans chaque jeu : oui")
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# chargement du dataset
data = load_breast_cancer()
X, y = data.data, data.target

# === CAS NORMAL ===
print("=== CAS NORMAL ===\n")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)

# === CAS LIMITE : val_size=0 ===
print("\n=== CAS LIMITE (val_size=0) ===\n")
try:
    split_train_val_test(X, y, val_size=0)
except ValueError as e:
    print(f"Erreur capturée proprement : {e}")

# === CAS ADVERSARIAL : dataset à 95/5 ===
print("\n=== CAS ADVERSARIAL (dataset 95/5) ===\n")
n = 500
y_deseq = np.array([0]*475 + [1]*25)
X_deseq = np.random.randn(n, 5)

X_tr, X_v, X_te, y_tr, y_v, y_te = split_train_val_test(X_deseq, y_deseq)
print("\n→ Sans stratify, le test pourrait n'avoir aucune classe minoritaire.")
print("  Avec stratify=y, les 5% sont bien répartis dans les 3 jeux")

=== CAS NORMAL ===

Train      : 341 lignes
Validation : 114 lignes
Test       : 114 lignes
Total      : 569 (original : 569)

  train  : classe 1 = 62.8%
  val    : classe 1 = 62.3%
  test   : classe 1 = 63.2%

Répartition des classes conservée dans chaque jeu : oui

=== CAS LIMITE (val_size=0) ===

Erreur capturée proprement : val_size doit être > 0

=== CAS ADVERSARIAL (dataset 95/5) ===

Train      : 300 lignes
Validation : 100 lignes
Test       : 100 lignes
Total      : 500 (original : 500)

  train  : classe 1 = 5.0%
  val    : classe 1 = 5.0%
  test   : classe 1 = 5.0%

Répartition des classes conservée dans chaque jeu : oui

Sans stratify, le test pourrait n'avoir aucune classe minoritaire.
Avec stratify=y, les 5% sont bien répartis dans les 3 jeux


In [8]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    """Évalue la stabilité d'un modèle par bootstrap.
    Pour chaque itération : tirer un échantillon AVEC REMISE de même taille
    que X, entraîner, évaluer sur les points NON tirés (out-of-bag).
    Doit renvoyer la liste des scores et afficher moyenne et écart-type.
    """
    rng = np.random.default_rng(random_state)
    scores = []
    n = len(X)

    for i in range(n_iterations):
        # tirer des indices avec remise
        indices_bootstrap = rng.choice(n, size=n, replace=True)

        # indices out-of-bag : points jamais tirés dans cet échantillon
        # ils n'ont pas servi à l'entraînement : jeu de test gratuit
        indices_oob = np.array(list(set(range(n)) - set(indices_bootstrap)))

        # cas rare : OOB vide → on skip cette itération
        if len(indices_oob) == 0:
            print(f"itération {i} : OOB vide, skippée")
            continue

        X_boot, y_boot = X[indices_bootstrap], y[indices_bootstrap]
        X_oob,  y_oob  = X[indices_oob],  y[indices_oob]

        # entraîner sur l'échantillon, scorer sur l'out-of-bag
        modele.fit(X_boot, y_boot)
        score = modele.score(X_oob, y_oob)
        scores.append(score)

    scores = np.array(scores)
    print(f"Score moyen sur {len(scores)} bootstraps : {scores.mean():.3f} (± {scores.std():.3f})")
    return scores

In [9]:
from sklearn.preprocessing import StandardScaler

# scaling d'abord (bonne pratique)
scaler_boot = StandardScaler()
X_scaled = scaler_boot.fit_transform(X)

modele_boot = RandomForestClassifier(n_estimators=50, random_state=42)

# === CAS NORMAL ===
print("=== CAS NORMAL (30 bootstraps) ===\n")
scores_boot = bootstrap_scores(modele_boot, X_scaled, y, n_iterations=30)

# === CAS LIMITE : replace=False (sans remise) ===
print("\n=== CAS LIMITE (sans remise replace=False) ===\n")
def bootstrap_scores_sans_remise(modele, X, y, n_iterations=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []
    n = len(X)
    for i in range(n_iterations):
        # sans remise = simple mélange, pas un vrai bootstrap
        indices = rng.choice(n, size=n, replace=False)
        milieu  = n // 2
        X_boot, y_boot = X[indices[:milieu]], y[indices[:milieu]]
        X_oob,  y_oob  = X[indices[milieu:]], y[indices[milieu:]]
        modele.fit(X_boot, y_boot)
        scores.append(modele.score(X_oob, y_oob))
    scores = np.array(scores)
    print(f"Score moyen SANS remise : {scores.mean():.3f} (± {scores.std():.3f})")
    return scores

scores_sans = bootstrap_scores_sans_remise(modele_boot, X_scaled, y)
print(f"\n→ Écart-type AVEC remise : {scores_boot.std():.3f}")
print(f"  Écart-type SANS remise : {scores_sans.std():.3f}")
print("  Sans remise = simple mélange, pas de vraie variété bootstrap")

# === CAS LIMITE : n_iterations=1 ===
print("\n=== CAS LIMITE (n_iterations=1) ===\n")
scores_1 = bootstrap_scores(modele_boot, X_scaled, y, n_iterations=1)
print("→ Un seul score, écart-type=0.000 : aucune stabilité mesurable")

=== CAS NORMAL (30 bootstraps) ===

Score moyen sur 30 bootstraps : 0.960 (± 0.015)

=== CAS LIMITE (sans remise replace=False) ===

Score moyen SANS remise : 0.955 (± 0.013)

→ Écart-type AVEC remise : 0.015
  Écart-type SANS remise : 0.013
  Sans remise = simple mélange, pas de vraie variété bootstrap

=== CAS LIMITE (n_iterations=1) ===

Score moyen sur 1 bootstraps : 0.910 (± 0.000)
→ Un seul score, écart-type=0.000 : aucune stabilité mesurable
